In [ ]:
import rawpy  # pip install rawpy
import cv2 # pip install opencv-python + pip install opencv-contrib-python
import glob
import math
import numpy as np
import os
from skimage.color import rgb2lab # pip3 install scikit-image
from PIL import Image, ImageCms
from colormath.color_objects import AdobeRGBColor, LabColor #pip3 install colormath // requires setuptools+networkx
from colormath.color_conversions import convert_color
import pandas as pd
from scipy import stats
import time
import matplotlib.pyplot as plt

In [ ]:
#Segmentation in Sensor display approach images with Fuzzy C-Means


import numpy as np
from fcmeans import FCM
from PIL import Image

t_inici = time.time()

folder = "Gamma_22"

files = [folder+"/papers_colors_v3_small_scene1_0347.png", folder+"/papers_colors_v3_small_scene2_0354.png", folder+"/papers_colors_v3_small_scene3_0364.png", folder+"/papers_colors_v3_small_scene4_0371.png"]


colors = ([420, 850], [550, 850], [690, 850], [850, 850])

df = pd.DataFrame() 
idx = 0

for f in files:
    
    if "coef" in f or "GT" in f: continue

    image_rgb = Image.open(f)          # read image (oarsman at https://omadson.github.io/photos/)
    N, M = image_rgb.size                                  # get the number of columns (N) and rows (M)
    image = image_rgb.convert(mode="HSV")
    
    clusters = 5
    
    arr_image = np.asarray(image)
    arr_image_hs = arr_image[:,:,0:2]
    
    X = (
        arr_image_hs                              # convert a PIL image to np array
        .reshape((N*M, 2))                             # reshape the image to convert each pixel to an instance of a data set
    )
    
    fcm = FCM(n_clusters=clusters)                           # create a FCM instance with 10 clusters
    fcm.fit(X)                                         # fit the model
    
    labeld_X = fcm.predict(X)                          # get the label of each data point
    transformed_X = fcm.centers[labeld_X]              # pixel quantization into the centers
    
    
    quatized_array = (
        transformed_X
        .astype('uint8')                               # convert data points into 8-bit unsigned integers
        .reshape((M, N, 2))                            # reshape image
    )
    
    #canal = np.zeros((M, N, 1))
    #np.insert(quatized_array, 1, 80, axis=2)
    #quantized_array = np.append(quatized_array, canal, axis=2)
    
    canal = np.full((M, N, 1), 200, dtype=np.uint8)  # brillantor constant (0-255)
    quantized_array = np.concatenate([quatized_array, canal], axis=2)
    array = quantized_array.astype('uint8')
    #array[:,:,0] = (array[:,:,0]/100)*255
    #print (array)
    
    
    array_rgb = cv2.cvtColor(array, cv2.COLOR_HSV2RGB)
    
    quatized_image_hsv = Image.fromarray(np.asarray(array_rgb))   # convert array into a PIL image object
    #quatized_image.save('images/oarsman_10colors.jpg') # save image

    image = cv2.imread(f)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    idx = 1

    t_clustering = time.time()
    print ("time clustering: ", t_clustering-t_inici)

    for c in colors:
    
        img_hsv = array   #ve de l'anterior
        img_rgb = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB)
        
        x = c[0]
        y = c[1]
            
        winsz = 2   #ajustar en f(x) del soroll/textura
        rect_winsz = 8
        
        color_roi = (255, 0, 0)
        
        img_roi = cv2.rectangle(img_rgb, (y-(rect_winsz//2), x-(rect_winsz//2)), (y+rect_winsz, x+rect_winsz), color_roi, 3)
       
        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
            
        h_roi = h_roi_np.flatten()
        #h = np.mean(h_roi)
        h = int(h_roi[1])
       
        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
            
        s_roi = s_roi_np.flatten()
        #s = np.mean(s_roi)
        s = int(s_roi[1])
    
    
        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)
            
    
        c1 = (h, s, 0) #per sota només resto 1/3 part del rang_S (perquè no estic mesurant a la zona més lluminosa) i el noise perquè rang S és assimètric
        c2 = (h, s, 255)
        
        print(c1,c2)
    
        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(image_rgb, image_rgb, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/Fuzzy_clust_"+str(clusters)+"_"+f[-15:-4]+"_"+str(idx)+".png"
        cv2.imwrite (result_name, result_bgr)
    
    
        idx += 1
            
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 3, 1)
        plt.imshow(image_rgb)
        plt.subplot(1, 3, 2)
        plt.imshow(img_roi)
        plt.subplot(1,3,3)
        plt.imshow(result)
        plt.show()

        t_final = time.time()
        print ("time detection: ", t_final-t_clustering)


In [ ]:
# FCM -- Precision, Recall, F1

import pandas as pd
import glob

folder = "../results"
GT_files = glob.glob("Gamma_22/*GT_2EV.png")
GT_files.sort()
results_files = glob.glob(folder+"/Fuzzy*.png")
results_files.sort()

df = pd.DataFrame()
idx = 0

for gt in GT_files:
        img_GT = cv2.imread(gt)
        img_GT = img_GT
        for res in results_files:
            if gt[-14:-11] in res and gt[-17] == res[-5]:
                result = cv2.imread(res)
                result = result
                FN_img = cv2.subtract(img_GT, result)
                FP_img = cv2.subtract(result, img_GT)
                TP_img = cv2.bitwise_and(result, img_GT)
                #TP_img = cv2.cvtColor(TP_img, cv2.COLOR_RGB2BGR)

                FN_img_g = cv2.cvtColor(FN_img, cv2.COLOR_BGR2GRAY)
                FP_img_g = cv2.cvtColor(FP_img, cv2.COLOR_BGR2GRAY)
                TP_img_g = cv2.cvtColor(TP_img, cv2.COLOR_BGR2GRAY)
                
                size = FN_img.shape
                total = size[0]*size[1]

                FN_im = FN_img_g.flatten()
                FP_im = FP_img_g.flatten()
                TP_im = TP_img_g.flatten()

                FN_list = FN_im[FN_im>0]
                FP_list = FP_im[FP_im>0]
                TP_list = TP_im[TP_im>0]

                FN = len(FN_list)
                FP = len(FP_list)
                TP = len(TP_list)

                print ("Total: ", total)
                print ("FN: ", FN)
                print ("FP: ", FP)
                print ("TP: ", TP)
                
                TN = total - TP - FP - FN

                print ("TN: ", TN)
                
                if TP == 0: 
                    prec = 0
                    recall = 0
                    F1 = 0
                elif TP != 0:
                    #print (res)
                    prec = TP / (TP+FP)
                    #print (prec)
                    recall = TP / (TP+FN)
                    #print (recall)
                    F1 = 2*(prec*recall)/(prec + recall)
                                
                #print ("Image: ", gt[58],    "Paper: ", gt[60:63])
                #print ("Classifier: ", clas)
                #print ("Precision: ", prec)
                #print ("Recall: ", recall)
                #print ("F1: ", F1)
                
                if "p1" in gt:
                    paper = 1
                elif "p2" in gt:
                    paper = 2
                elif "p3" in gt:
                    paper = 3
                elif "p4" in gt:
                    paper = 4
                
                df.at[idx, "File"] = res
                df.at[idx, "Image"] = gt[-23]
                df.at[idx, "Paper"] = paper
                df.at[idx, "Method"] = "Fuzzy C-Means"
                df.at[idx, "TP"] = TP
                df.at[idx, "FP"] = FP
                df.at[idx, "FN"] = FN
                df.at[idx, "Precision"] = prec
                df.at[idx, "Recall"] = recall
                df.at[idx, "F1"] = F1
                
                idx += 1

print(df)
df.to_excel("../results/FCM_F1.xlsx") 